# 06 · Linked List

> Nodi collegati da puntatori `next`: niente indici, ma inserimenti e rimozioni in `O(1)` se hai il nodo. Quasi tutto si riduce a quattro mosse: nodo **dummy**, **inversione**, **fast/slow**, **merge**.
> Idee per i singoli problemi: [Linked list e heap › Linked list](../Fundamentals/Linked%20list%20e%20heap.md#linked-list)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Reverse Linked List | tre puntatori `prev`, `cur`, `nxt` | assegnazione multipla |
| Merge Two Sorted Lists | dummy + coda che avanza | `dummy = ListNode()`, `tail.next = ...`, `return dummy.next` |
| Linked List Cycle | fast/slow (Floyd) | `while fast and fast.next` |
| Reorder List | metà (fast/slow) + inversione della seconda metà + merge alternato | combinazione delle tre mosse |
| Remove Nth Node From End | due puntatori distanziati di `n` | dummy per gestire la rimozione della testa |
| Copy List with Random Pointer | mappa nodo originale → copia | `dict` con oggetti come chiavi, `old_to_new.get(node)` |
| Add Two Numbers | cifre al contrario, riporto | `carry, digit = divmod(total, 10)` |
| Find the Duplicate Number | l'array come lista: `i → nums[i]`; Floyd trova l'ingresso del ciclo | due fasi di fast/slow |
| LRU Cache | dict + lista doppiamente collegata (o `OrderedDict`) | `move_to_end`, `popitem(last=False)` |
| Merge K Sorted Lists | heap delle teste | `heappush(h, (node.val, i, node))` |
| Reverse Nodes in K-Group | conta k nodi, inverti il gruppo, ricollega | funzione di supporto, puntatore al nodo prima del gruppo |

## Il nodo e due funzioni di comodo

Su LeetCode `ListNode` è già definito (non ridefinirlo). Qui lo definiamo, insieme a due helper per creare e stampare liste mentre fai prove.

In [1]:
from typing import Optional

class ListNode:
    def __init__(self, val: int = 0, next: "Optional[ListNode]" = None):
        self.val = val
        self.next = next

def build(values: list[int]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def to_list(head: Optional[ListNode]) -> list[int]:
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

print(to_list(build([1, 2, 3])))  # -> [1, 2, 3]

[1, 2, 3]


## Nodo dummy

Un nodo fittizio prima della testa elimina il caso speciale "lista vuota" o "devo cambiare la testa". Costruisci dopo `dummy` e restituisci `dummy.next`. È lo stesso trucco di `build` qui sopra.

In [2]:
# continua dall'esempio precedente: rimuove tutti i nodi con un valore, testa compresa
def remove_value(head: Optional[ListNode], value: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    cur = dummy
    while cur.next:
        if cur.next.val == value:
            cur.next = cur.next.next  # salta il nodo
        else:
            cur = cur.next
    return dummy.next

print(to_list(remove_value(build([7, 1, 7, 2]), 7)))  # -> [1, 2]

[1, 2]


## Inversione

Salva `next` prima di sovrascriverlo, punta indietro, avanza. L'assegnazione multipla di Python valuta prima tutto il lato destro, quindi si può scrivere in una riga, ma la forma estesa è più leggibile.

In [3]:
# continua dall'esempio precedente
def reverse(head: Optional[ListNode]) -> Optional[ListNode]:
    prev, cur = None, head
    while cur:
        nxt = cur.next   # 1. salva il resto
        cur.next = prev  # 2. punta indietro
        prev = cur       # 3. avanza entrambi
        cur = nxt
    return prev  # nuova testa

print(to_list(reverse(build([1, 2, 3, 4]))))  # -> [4, 3, 2, 1]

[4, 3, 2, 1]


## Fast / slow

`fast` fa due passi, `slow` uno. Quando `fast` arriva in fondo, `slow` è a metà. Se c'è un ciclo, i due si incontrano. La condizione `while fast and fast.next` protegge entrambi gli accessi.

In [4]:
# continua dall'esempio precedente
def middle(head: Optional[ListNode]) -> Optional[ListNode]:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print(middle(build([1, 2, 3, 4, 5])).val, middle(build([1, 2, 3, 4])).val)  # -> 3 3

# ciclo: l'ultimo nodo punta al secondo
head = build([1, 2, 3, 4])
head.next.next.next.next = head.next
slow = fast = head
while fast and fast.next:
    slow, fast = slow.next, fast.next.next
    if slow is fast:  # `is`: stesso oggetto, non stesso valore
        print("ciclo trovato")  # -> ciclo trovato
        break

3 3
ciclo trovato


**Ingresso del ciclo (Floyd, fase 2):** dopo l'incontro, riporta un puntatore alla testa e muovi entrambi di un passo: si incontrano all'ingresso del ciclo. In *Find the Duplicate Number* la "lista" è `i → nums[i]` e l'ingresso è il duplicato.

In [5]:
nums = [1, 3, 4, 2, 2]  # indici 0..4, valori 1..4: un duplicato
slow = fast = 0
while True:
    slow = nums[slow]
    fast = nums[nums[fast]]
    if slow == fast:
        break
print("incontro in", slow)  # -> incontro in 4 (dentro il ciclo, non per forza l'ingresso)

incontro in 4


## Due puntatori distanziati

Per l'n-esimo dalla fine: porta `fast` avanti di `n` passi, poi muovi entrambi finché `fast` arriva in fondo. Partendo da `dummy`, `slow` si ferma sul nodo **prima** di quello da togliere.

In [6]:
# continua dall'esempio precedente
head = build([10, 20, 30, 40])
dummy = ListNode(0, head)
slow = fast = dummy
for _ in range(2):  # n = 2
    fast = fast.next
while fast.next:
    slow, fast = slow.next, fast.next
print(slow.val, slow.next.val)  # -> 20 30: 30 è il secondo dalla fine

20 30


## Oggetti come chiavi di dict

I nodi sono hashable per identità: puoi mapparli alle loro copie. Utile quando la lista ha puntatori extra (`random`) che possono puntare a nodi non ancora copiati: prima crei tutte le copie, poi colleghi.

In [7]:
# continua dall'esempio precedente
head = build([1, 2, 3])
old_to_new: dict[ListNode, ListNode] = {}
cur = head
while cur:
    old_to_new[cur] = ListNode(cur.val)  # prima passata: solo i nodi
    cur = cur.next
print(len(old_to_new), old_to_new.get(None))  # -> 3 None: .get evita il caso next/random = None

3 None


## Riporto con `divmod`

In [8]:
carry = 0
for a, b in [(9, 9), (9, 1)]:  # cifre meno significative per prime
    carry, digit = divmod(a + b + carry, 10)
    print(digit, carry)  # -> 8 1 / 1 1
# alla fine, se carry > 0, serve un nodo in più

8 1
1 1


## LRU: `OrderedDict`

`OrderedDict` ricorda l'ordine di inserimento e permette di spostare una chiave in fondo in `O(1)`. In colloquio spesso ti chiedono la versione con dict + lista doppiamente collegata (nodi con `prev` e `next`, due sentinelle `head`/`tail`), che è quello che `OrderedDict` fa internamente.

In [9]:
from collections import OrderedDict

cache: OrderedDict[str, int] = OrderedDict()
cache["MUG"] = 990
cache["CAP"] = 1200
cache["TSHIRT"] = 1500

cache.move_to_end("MUG")          # appena usato: diventa il più recente
print(list(cache))                # -> ['CAP', 'TSHIRT', 'MUG']
print(cache.popitem(last=False))  # -> ('CAP', 1200): rimuove il meno recente

['CAP', 'TSHIRT', 'MUG']
('CAP', 1200)


## Heap di nodi

Per unire `k` liste ordinate, tieni in un min-heap la testa di ogni lista. I nodi non sono confrontabili: aggiungi un indice come spareggio nella tupla, così Python non arriva mai a confrontare due `ListNode`.

In [10]:
# continua dall'esempio precedente
import heapq

lists = [build([1, 4]), build([1, 3]), build([2])]
heap = [(node.val, i, node) for i, node in enumerate(lists) if node]
heapq.heapify(heap)
val, i, node = heapq.heappop(heap)
print(val, i)  # -> 1 0: a parità di valore decide l'indice
# poi: if node.next: heappush(heap, (node.next.val, i, node.next))

1 0


## Errori comuni

- Perdere il resto della lista: salva `nxt = cur.next` prima di modificare `cur.next`.
- Accedere a `fast.next.next` senza aver controllato `fast.next`.
- Confrontare nodi con `==` quando intendi lo stesso oggetto: usa `is`.
- Dopo `reverse`, la vecchia testa è la coda e il suo `next` è `None`: ricollega i pezzi (Reorder List, K-Group).
- `heappush` di `(val, node)` con valori uguali: `TypeError` perché confronta i nodi.